# Agent Valley Training Notebook

This project implements CPU-safe RL training over the real Agent Valley environment using tabular Q-learning, a small neural policy, and a compact GRPO-style grouped reward optimization loop. It is not full LLM fine-tuning with TRL/Unsloth.

## Setup

Run this notebook from the repository root, or update `PROJECT_ROOT` below. In Colab, upload or clone the project first, then install `requirements.txt`.

In [ ]:
from pathlib import Path
import json
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'env').exists():
    PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
PROJECT_ROOT

## Environment Reset And Step Demo

In [ ]:
from env.environment import AgentValleyEnv
from env.action_space import index_to_action

env = AgentValleyEnv(difficulty='easy', seed=42)
obs = env.reset()
action = index_to_action(0).model_dump(mode='json')
next_obs, reward, done, info = env.step(action)
print('observation keys:', sorted(obs.keys()))
print('action:', action)
print('reward:', reward, 'done:', done)
print('reward breakdown:', info['reward_breakdown'])

## Reward Breakdown Demo

In [ ]:
from baseline_eval import RuleBasedAgent

env = AgentValleyEnv(difficulty='easy', seed=42)
agent = RuleBasedAgent()
obs = env.reset()
done = False
while not done:
    action = agent.act(obs)
    obs, reward, done, info = env.step(action)
print(json.dumps(info['episode_result'], indent=2))

## Q-Learning Training Demo

In [ ]:
from training.q_learning import QLearningConfig, QLearningTrainer

q_trainer = QLearningTrainer(QLearningConfig(episodes=2, difficulty='easy', seed=7, reset_metrics=True))
q_metrics = q_trainer.train()
q_metrics[-1]

## Neural Policy Training Demo

In [ ]:
from training.train_neural_policy import NeuralPolicyConfig, NeuralPolicyTrainer

np_trainer = NeuralPolicyTrainer(NeuralPolicyConfig(episodes=2, difficulty='easy', seed=11, reset_metrics=True))
np_metrics = np_trainer.train()
np_metrics[-1]

## GRPO-Style Training Demo

In [ ]:
from training.grpo_train import GRPOConfig, GRPOTrainer

grpo_trainer = GRPOTrainer(GRPOConfig(episodes=2, difficulty='easy', seed=13, group_size=3, reset_metrics=True))
grpo_metrics = grpo_trainer.train()
grpo_metrics[-1]

## Load Metrics JSONL

In [ ]:
def read_jsonl(path):
    path = PROJECT_ROOT / path
    if not path.exists():
        return []
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

metrics = {
    'q_learning': read_jsonl('artifacts/q_learning/metrics.jsonl'),
    'neural_policy': read_jsonl('artifacts/neural_policy/metrics.jsonl'),
    'grpo': read_jsonl('artifacts/grpo/metrics.jsonl'),
}
{mode: len(rows) for mode, rows in metrics.items()}

## Plot Reward And Loss Curves

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print('matplotlib is not installed; run scripts/generate_training_plots.py for PNG artifacts.')

if plt:
    for mode, rows in metrics.items():
        xs = [row.get('episode', i + 1) for i, row in enumerate(rows)]
        ys = [row.get('total_reward') for row in rows]
        if xs and ys:
            plt.plot(xs, ys, label=mode)
    plt.title('Backend total reward')
    plt.xlabel('Episode')
    plt.ylabel('Total reward')
    plt.legend()
    plt.show()

    for mode, rows in metrics.items():
        points = [(row.get('update_step', row.get('episode', i + 1)), row.get('policy_loss')) for i, row in enumerate(rows) if isinstance(row.get('policy_loss'), (int, float))]
        if points:
            xs, ys = zip(*points)
            plt.plot(xs, ys, label=mode)
    plt.title('Backend policy loss')
    plt.xlabel('Episode/update step')
    plt.ylabel('Policy loss')
    plt.legend()
    plt.show()

## Evaluate Latest Policy Checkpoint

In [ ]:
from training.policy_runtime import evaluate_policy

evaluate_policy(mode='grpo', difficulty='easy', episodes=1, seed=42)

## Frontend / Backend Metric Flow

The React dashboard does not calculate training curves locally. It polls `/api/training/status`, `/api/training/metrics`, and `/api/training/latest`; those endpoints read the same backend JSONL files loaded above.

## Multi-Agent Extension

`env.multi_agent_env.MultiAgentValleyEnv` wraps the base environment with four simultaneous role agents. See `notebooks/multi_agent_training.ipynb` for a reset/step demo and a tiny multi-agent GRPO-style run.